#### Objetivo

- Esse notebook consiste em concatenar os dados dos eventos de todas as partidas das competições entre todas as suas temporadas. 

    A ideia é de analisar principalmente a disponibilidade dos dados de tracking e entender qual/quais temporadas sugerem estar mais consistentes de serem utilizadas.

In [33]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

In [34]:
competitions = [competition for competition in os.listdir(str(Path().resolve().parent.parent / "data" / "events"))]
competitions

['1', '42']

In [35]:
competitions_seasons = {
    competitions[id]: [season for season in os.listdir(str(Path().resolve().parent.parent / "data" / "events" / competitions[id]))]
    for id in range(len(competitions))
}
competitions_seasons

{'1': ['2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025'],
 '42': ['2023', '2024', '2025']}

In [36]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

In [37]:
def get_competition_seasons_parquet_file_paths(competition, seasons):

    competition_seasons_parquet_file_paths = []

    for season in seasons:

        events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events_parsed" / competition / season)

        competition_seasons_parquet_file_paths.extend(get_season_events_parquet_file_paths(events_competition_season_folder_path))
    
    return competition_seasons_parquet_file_paths

In [38]:
competitions_seasons_events_parquet_file_paths = []

for competition, seasons in competitions_seasons.items():

    competitions_seasons_events_parquet_file_paths.extend(get_competition_seasons_parquet_file_paths(competition, seasons))

#competitions_seasons_events_parquet_file_paths

In [39]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("season_events_sanity")
    .getOrCreate()
)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df = spark.read.parquet(*competitions_seasons_events_parquet_file_paths)

## Sanity Check dos dados dos eventos/partidas em geral

In [40]:
df.printSchema()

root
 |-- eventId: string (nullable = true)
 |-- competitionId: long (nullable = true)
 |-- gameId: long (nullable = true)
 |-- season: string (nullable = true)
 |-- period: long (nullable = true)
 |-- periodDescription: string (nullable = true)
 |-- eventType: string (nullable = true)
 |-- eventTypeDescription: string (nullable = true)
 |-- startGameClock: long (nullable = true)
 |-- startFormattedGameClock: string (nullable = true)
 |-- homeTeam: boolean (nullable = true)
 |-- eventPlayerId: long (nullable = true)
 |-- eventPlayerName: string (nullable = true)
 |-- eventTeamId: long (nullable = true)
 |-- eventTeamName: string (nullable = true)
 |-- homePlayers_parsed: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- x: float (nullable = true)
 |    |    |-- y: float (nullable = true)
 |    |    |-- player: struct (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |-- awayPlayers_parsed: array (nullable = true)
 |    |-- elemen

In [41]:
print(f'Quantidade total de eventos: {df.select('eventId').count()}')
print(f'Quantidade de eventos distintos: {df.select('eventId').distinct().count()}')

print(f'Quantidade de eventos duplicados: {df.select('eventId').count() - df.select('eventId').distinct().count()}')

Quantidade total de eventos: 7580243
Quantidade de eventos distintos: 7578205
Quantidade de eventos duplicados: 2038


In [42]:
(
    df
    .groupBy('gameId')
    .agg(F.count(F.col('eventId')).alias('qtd_eventos'))
    .agg(F.round(F.avg(F.col('qtd_eventos')), 2).alias('Quantidade média de eventos por partida'))
).show()

+---------------------------------------+
|Quantidade média de eventos por partida|
+---------------------------------------+
|                                2505.04|
+---------------------------------------+



- Existem uma quantidade média de ~2500 eventos por partida dentre todas as temporadas.

In [43]:
(
    df
    .groupBy('eventTypeDescription')
    .agg(F.count(F.col('eventId')).alias('Quantidade de Eventos'))
    .sort('Quantidade de Eventos', ascending=False)
    .show(truncate=False)
)

+--------------------------------------------------------------+---------------------+
|eventTypeDescription                                          |Quantidade de Eventos|
+--------------------------------------------------------------+---------------------+
|A possession with a player on the ball                        |3591134              |
|Pass                                                          |2836374              |
|Challenge                                                     |546751               |
|Clearance                                                     |138419               |
|Rebound                                                       |132151               |
|Cross                                                         |123509               |
|Shot                                                          |81056                |
|Ball Carry                                                    |79147                |
|Touch Carry                               

- Existem diversos tipos de eventos, principalmente ofensivos e defensivos. 
- Alguns eventos tem baixa volumetria e/ou não contemplam o objetivo das análises e modelagens, como 'First half kick off', 'Second half kick off', eventos inválidos sem descrição como 'Unknown', substituições como 'Substitution', 'Player comes off the pitch' e 'Ball hits the woodwork or corner flag and comes back into play'.

In [44]:
(
    df
    .groupBy('period')
    .agg(F.count(F.col('eventId')).alias('Quantidade de Eventos'))
    .sort('Quantidade de Eventos', ascending=False)
    .show(truncate=False)
)

+------+---------------------+
|period|Quantidade de Eventos|
+------+---------------------+
|1     |3823677              |
|2     |3756566              |
+------+---------------------+



- Os eventos estão distribuidos em apenas 2 periodos, o que é correto visto que todas as competições são de pontos corridos e sem prorrogação.

In [45]:
# variável que indica o time com a posse
df.groupBy('homeTeam').count().show()
df.filter(F.col('homeTeam').isNull()).groupBy('homeTeam', 'eventTypeDescription').count().show()

+--------+-------+
|homeTeam|  count|
+--------+-------+
|    NULL|  51961|
|    true|3825148|
|   false|3703134|
+--------+-------+

+--------+--------------------+-----+
|homeTeam|eventTypeDescription|count|
+--------+--------------------+-----+
|    NULL|A possession with...|25156|
|    NULL|           Challenge|24762|
|    NULL|             Unknown| 1598|
|    NULL|         Touch Carry|  119|
|    NULL|          Ball Carry|  139|
|    NULL|             Rebound|   11|
|    NULL|                Pass|  169|
|    NULL|               Cross|    2|
|    NULL|Ball hits the woo...|    1|
|    NULL|           Clearance|    1|
|    NULL|        Substitution|    2|
|    NULL|                Shot|    1|
+--------+--------------------+-----+



- Existem 51961 eventos sem posse e os dados mostram serem a grande maioria como disputas. Vamos removê-los, por enquanto, mas uma alternativa seria associar um homeTeam a ele vendo quem ficou com posse no evento seguinte.

## Sanity Check dos Dados de tracking dos eventos

In [46]:
df = df.withColumns({

    # Confere se nos dados de tracking do mandante não é lista vazia e tem pelo menos um x,y preenchidos entre os jogadores (True/False convertido para binário)
    "has_tracking_home":
    exists(
        col("homePlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    ).cast("int"),

    # Confere se nos dados de tracking do adversário não é lista vazia e tem pelo menos um x,y preenchidos entre os jogadores (True/False convertido para binário)
    "has_tracking_away":
    exists(
        col("awayPlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    ).cast("int"),

    # Confere se nos dados de tracking do mandante não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_home":
    ((size(col("homePlayers_parsed")) > 0) & 
    forall(
        col("homePlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Confere se nos dados de tracking do adversário não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_away":
    ((size(col("awayPlayers_parsed")) > 0) & 
    forall(
        col("awayPlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    "all_balls": 
    ((size(col("balls_parsed")) > 0) & 
    forall(
        col("balls_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Traz a quantidade de dicionários de cada evento para saber se tem 11 jogadores do time mandante e adversário
    "len_tracking_home": size(col("homePlayers_parsed")),
    "len_tracking_away": size(col("awayPlayers_parsed"))
}
)

In [47]:
df.show(5)

+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+-------------+---------------+-----------+---------------+--------------------+--------------------+--------------------+--------------------+-----------------------+-----------------------+-----------------+-----------------+-----------------+-----------------+---------+-----------------+-----------------+
|             eventId|competitionId|gameId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|eventPlayerId|eventPlayerName|eventTeamId|  eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      details_parsed|eventSubTypeDescription|eventOutcomeDescription|has_tracking_home|has_tracking_away|all_tracking_home|all_tracking_away|all_balls|len_tracking_home|len_tracking_away|
+--------------------+-------------+------+---------+------+--

In [48]:
df_agg_games = (
    df.groupBy("gameId")
      .agg(
          max("competitionId").alias("competitionId"),
          max("season").alias("season"),
          round(F.mean("has_tracking_home"), 3).alias("has_tracking_home_percent"),
          round(F.mean("has_tracking_away"), 3).alias("has_tracking_away_percent"),
          round(F.mean("all_tracking_home"), 3).alias("all_tracking_home_percent"),
          round(F.mean("all_tracking_away"), 3).alias("all_tracking_away_percent"),
          round(F.mean("all_balls"), 3).alias("all_balls_percent"),
          round(F.mean("len_tracking_home"), 3).alias("tracking_home_len"),
          round(F.mean("len_tracking_away"), 3).alias("tracking_away_len"),
      )
)

df_agg_games.cache()

DataFrame[gameId: bigint, competitionId: bigint, season: string, has_tracking_home_percent: double, has_tracking_away_percent: double, all_tracking_home_percent: double, all_tracking_away_percent: double, all_balls_percent: double, tracking_home_len: double, tracking_away_len: double]

In [49]:
df_agg_games.show(5)

+------+-------------+---------+-------------------------+-------------------------+-------------------------+-------------------------+-----------------+-----------------+-----------------+
|gameId|competitionId|   season|has_tracking_home_percent|has_tracking_away_percent|all_tracking_home_percent|all_tracking_away_percent|all_balls_percent|tracking_home_len|tracking_away_len|
+------+-------------+---------+-------------------------+-------------------------+-------------------------+-------------------------+-----------------+-----------------+-----------------+
|   474|            1|2020-2021|                    0.998|                    0.998|                    0.998|                    0.998|            0.942|           10.979|           10.979|
|  4590|            1|2022-2023|                      1.0|                      1.0|                      1.0|                      1.0|            0.974|             11.0|             11.0|
| 12568|           42|     2023|             

In [50]:
print('Quantidade de partidas sem nenhum dado de tracking do mandante:', df_agg_games.filter(col('has_tracking_home_percent') == 0).count())
print('Quantidade de partidas sem nenhum dado de tracking do adversário:', df_agg_games.filter(col('has_tracking_home_percent') == 0).count())

print('Quantidade de partidas que possui eventos com dados de tracking incomuns (diferente de 11) do mandante:', df_agg_games.filter(col('tracking_home_len') != 11).count())
print('Quantidade de partidas que possui eventos com dados de tracking incomuns (diferente de 11) do adversário:', df_agg_games.filter(col('tracking_away_len') != 11).count())

print('Quantidade de partidas que possui eventos com dados de tracking faltando dos 22 jogadores:', df_agg_games.filter((col('all_tracking_home_percent') < 1) | (col('all_tracking_away_percent') < 1)).count())

print('Quantidade de partidas que possui eventos com dados de tracking faltando da bola:', df_agg_games.filter(col('all_balls_percent') < 1).count())
print('Quantidade de partidas sem nenhum dado de tracking da bola:', df_agg_games.filter(col('all_balls_percent') == 0).count())

Quantidade de partidas sem nenhum dado de tracking do mandante: 52
Quantidade de partidas sem nenhum dado de tracking do adversário: 52
Quantidade de partidas que possui eventos com dados de tracking incomuns (diferente de 11) do mandante: 988
Quantidade de partidas que possui eventos com dados de tracking incomuns (diferente de 11) do adversário: 1030
Quantidade de partidas que possui eventos com dados de tracking faltando dos 22 jogadores: 364
Quantidade de partidas que possui eventos com dados de tracking faltando da bola: 3026
Quantidade de partidas sem nenhum dado de tracking da bola: 52


- Existem 52 partidas que não possuem nenhum dado de tracking para o time mandante ou adversário.
- Existem 208 partidas que possuem dados de tracking a mais ou a menos de jogadores do time mandante e 224 partidas do time adversário.
- Existem 364 partidas que não possuem dados de tracking de todos os 22 jogadores em campo.
- Todas as partidas das temporadas há dados faltando da bola. Em 52 partidas não tem nenhum dado de tracking da bola.

In [51]:
df_agg_games.filter((col('has_tracking_home_percent') == 0) | (col('has_tracking_away_percent') == 0)).groupBy('competitionId', 'season').count().show()

+-------------+------+-----+
|competitionId|season|count|
+-------------+------+-----+
|           42|  2025|   52|
+-------------+------+-----+



- Dentre as 52 partidas com dados de tracking faltando, todas são da temporada de 2025 no Brasileirão.

In [52]:
df_agg_games.filter((col('all_balls_percent') == 0)).groupBy('competitionId', 'season').count().show()

+-------------+------+-----+
|competitionId|season|count|
+-------------+------+-----+
|           42|  2025|   52|
+-------------+------+-----+



- As 52 partidas que não possui dados de tracking da bola também são da temporada de 2025 no Brasileirão, sugerindo ser a mesma que faltou dados de tracking dos jogadores.

In [53]:
df_agg_games.filter((col('tracking_home_len') != 11) | (col('tracking_home_len') != 11)).groupBy('competitionId', 'season').count().orderBy('competitionId', 'season').show()

+-------------+---------+-----+
|competitionId|   season|count|
+-------------+---------+-----+
|            1|2020-2021|  278|
|            1|2021-2022|   28|
|            1|2022-2023|   36|
|            1|2023-2024|  129|
|            1|2024-2025|  107|
|           42|     2023|  104|
|           42|     2024|  109|
|           42|     2025|  197|
+-------------+---------+-----+



- Apesar de existir registros faltando de tracking de jogadores em alguns eventos, são poucos registros de forma geral. 
- Porém, o ano de 2025 do Brasileirão apresenta uma quantidade elevada de 144 registros com ausência de dados de tracking de algum jogador.

In [54]:
df_agg_games.filter((col('all_tracking_home_percent') < 1) | (col('all_tracking_away_percent') < 1)).groupBy('competitionId', 'season').count().orderBy('competitionId', 'season').show()

+-------------+---------+-----+
|competitionId|   season|count|
+-------------+---------+-----+
|            1|2020-2021|  200|
|            1|2021-2022|    5|
|            1|2022-2023|    3|
|            1|2023-2024|   28|
|            1|2024-2025|   10|
|           42|     2023|   25|
|           42|     2024|   27|
|           42|     2025|   66|
+-------------+---------+-----+



- Pode-se notar que todas as temporadas das duas competições possuem uma certa quantidade de dados faltando para os 22 jogadores. 
- Tratando-se de Premier League, as temporadas com a menor quantidade de dados de tracking faltando seriam as de 2021-2022, 2022-2023 e 2024-2025.
- Já sobre o Brasileirão, as temporadas de 2023 e 2024 possuem uma quantidade um pouco elevada, mas seriam as mais adequadas para se trabalhar. A temporada de 2025 possui 109 eventos faltando informações de tracking, sendo que 52 delas faltam 100% dos dados de tracking.
- Para a análise de uma temporada em específico, é recomendado **começar pela de 2022-2023 da Premier League** por apresentar o menor número de registros faltantes.

In [55]:
df_agg_games.filter(col('all_balls_percent') < 1).groupBy('competitionId', 'season', 'GameId').count().orderBy('GameId', ascending=False).show()

+-------------+------+------+-----+
|competitionId|season|GameId|count|
+-------------+------+------+-----+
|           42|  2025| 37280|    1|
|           42|  2025| 37279|    1|
|           42|  2025| 37278|    1|
|           42|  2025| 37277|    1|
|           42|  2025| 37276|    1|
|           42|  2025| 37275|    1|
|           42|  2025| 37274|    1|
|           42|  2025| 37273|    1|
|           42|  2025| 37272|    1|
|           42|  2025| 37271|    1|
|           42|  2025| 37270|    1|
|           42|  2025| 37269|    1|
|           42|  2025| 37268|    1|
|           42|  2025| 37267|    1|
|           42|  2025| 37266|    1|
|           42|  2025| 37265|    1|
|           42|  2025| 37264|    1|
|           42|  2025| 37263|    1|
|           42|  2025| 37262|    1|
|           42|  2025| 37261|    1|
+-------------+------+------+-----+
only showing top 20 rows


- Toda partida possui exatamente 1 dado faltando da bola, então será necessário imputação independente da temporada escolhida.